# 01 — Setup Environment · Banana Ripeness Detection (YOLO26s)

**หน้าที่ของ notebook นี้**
1. ตรวจเครื่องที่รันจริง (Python / OS / CPU / RAM / Disk / Kernel)
2. ติดตั้ง-verify `ultralytics` และ **assert ว่ารองรับ YOLO26 จริง** (end-to-end head, MuSGD, ProgLoss, `nms=`)
3. ตรวจ CUDA → เลือก `DEVICE` (fallback CPU)
4. สร้างโครงโฟลเดอร์โปรเจกต์ + เขียน `configs/project.yaml` (single source of truth ให้ nb02–07)
5. ดาวน์โหลด `yolo26s.pt` + smoke test 2 โหมด (NMS default vs NMS-free)
6. บันทึก `reports/env_report.json` + `requirements.txt`

> รันจากโฟลเดอร์ `GROUP NI YOD YIAM MAK MAK/YOLOV26/` ด้วย kernel **ML Environment (ds69)** — ทุก cell รันต่อเนื่องได้โดยไม่ต้องแก้มือ


In [ ]:
# === CELL 1 === CONFIG กลาง + Seed
# ทำอะไร : รวม path / hyper-parameter / class taxonomy ของทั้งโปรเจกต์ไว้ใน dict เดียว แล้ว set seed ทุก library
# ทำไม   : notebook 02–07 จะอ่านค่าจาก configs/project.yaml ที่ cell นี้สร้าง → ไม่มี hardcode path กระจัดกระจาย
#          และ seed=42 ตั้งแต่ต้นทำให้ทุกขั้น (split / train / eval) ทำซ้ำได้ (reproducible)
import os, sys, json, random, platform, shutil, subprocess, time, importlib, importlib.util
from datetime import datetime
from pathlib import Path

# JupyterLab ตั้ง cwd = โฟลเดอร์ที่ notebook อยู่ → ต้องเป็น .../GROUP NI YOD YIAM MAK MAK/YOLOV26/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.upper() not in {"YOLOV26", "YOLO26"}:
    print(f"[WARN] cwd = {PROJECT_ROOT}\n       ไม่ใช่โฟลเดอร์ YOLOV26 — ให้เปิด notebook นี้จากโฟลเดอร์โปรเจกต์ก่อนรันต่อ")

CONFIG = {
    "project_name": "banana_ripeness_yolo26s",
    "project_root": str(PROJECT_ROOT),
    "created_by": "01_setup_env.ipynb",
    "seed": 42,

    # --- Model & training spec (MASTER PROMPT ข้อ 6) ---
    "model": {
        "base_weights": "yolo26s.pt",   # pretrained COCO → fine-tune
        "imgsz": 640,
        "epochs": 150,
        "patience": 30,
        "batch": -1,                    # -1 = AutoBatch (ultralytics เลือก batch ให้พอดี VRAM ~60%)
        "optimizer": "MuSGD",           # ต้องระบุตรง ๆ: 'auto' จะเลือก MuSGD เฉพาะเมื่อ iterations > 10k
        "tracker": "bytetrack.yaml",    # default ของ ultralytics 8.4 คือ tracktrack.yaml → ต้อง override ใน nb07
        "deploy_nms": False,            # nms=False = NMS-free one-to-one head (จุดเด่น YOLO26) ใช้ตอน deploy
    },

    # --- Unified taxonomy 4 คลาส (index ต้องตรงนี้เสมอ) ---
    "classes": {0: "unripe", 1: "semi_ripe", 2: "ripe", 3: "overripe"},
    "classes_th": {0: "ยังไม่สุก", 1: "สุกปานกลาง", 2: "สุก", 3: "สุกงอมเกินไป"},
    "class_colors_rgb": {                # ใช้วาด bbox ใน nb07 (แปลงเป็น BGR ตอนใช้กับ OpenCV)
        0: [0, 170, 0],      # unripe    = เขียว
        1: [154, 205, 50],   # semi_ripe = เหลืองอมเขียว
        2: [255, 215, 0],    # ripe      = เหลือง
        3: [139, 69, 19],    # overripe  = น้ำตาล
    },

    # --- Dataset sources (ใช้ใน nb02) ---
    "datasets": {
        "mendeley": {
            "name": "Annotated Banana Ripeness Stage Dataset",
            "id": "56c75hvm7r", "version": 1,
            "url": "https://data.mendeley.com/datasets/56c75hvm7r/1",
            "expected_images": 673,
            "class_map": None,   # กำหนดใน nb02 หลังอ่าน metadata/ชื่อคลาสจริง (ห้ามเดา)
        },
        "roboflow": {
            "name": "BANAna Ripening Process",
            "workspace": "fruit-ripening", "project": "banana-ripening-process", "version": 1,
            "url": "https://universe.roboflow.com/fruit-ripening/banana-ripening-process",
            "expected_images": 7546,
            "class_map": {"freshunripe": 0, "freshripe": 1, "ripe": 2, "overripe": 3},
        },
        "expected_total": 8219,
        "split_ratio": {"train": 0.80, "valid": 0.10, "test": 0.10},
        "expected_split": {"train": 6575, "valid": 822, "test": 822},
    },

    # --- โครงโฟลเดอร์ (relative ต่อ project_root) ---
    "dirs": {
        "configs": "configs",
        "data_raw": "data/raw",              # dataset ดิบที่ดาวน์โหลด (nb02)
        "data_interim": "data/interim",      # หลัง merge + remap class (nb02)
        "data_processed": "data/processed",  # หลัง cleaning (nb03)
        "datasets": "datasets/banana",       # split train/valid/test + data.yaml (nb04)
        "weights": "weights",
        "runs": "runs",                      # runs/detect/<exp_name>/ (nb05)
        "figures": "figures",                # ทุกกราฟ .png dpi=300 สำหรับเล่มรายงาน
        "reports": "reports",                # csv / md / json
        "videos_in": "videos/input",
        "videos_out": "videos/output",
        "src": "src",                        # helper module ใช้ร่วมกันหลาย notebook
        "logs": "logs",
    },
}

DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]

def set_seed(seed: int) -> None:
    # set seed ให้ random / numpy / torch (ถ้ามี) + PYTHONHASHSEED
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed(SEED)
ENV_REPORT = {"notebook": "01_setup_env", "started_at": datetime.now().isoformat(timespec="seconds")}
print(f"[OK] PROJECT_ROOT = {PROJECT_ROOT}")
print(f"[OK] seed = {SEED} | classes = {CONFIG['classes']}")


In [ ]:
# === CELL 2 === ตรวจระบบ: Python / OS / CPU / RAM / Disk / Kernel
# ทำอะไร : เก็บข้อมูลเครื่องที่รันจริง + เตือนถ้าพื้นที่ดิสก์ไม่พอ + ยืนยันว่ารันใน kernel ds69
# ทำไม   : dataset ~8.2k ภาพ + weights + runs ใช้พื้นที่หลาย GB และเล่มรายงานต้องระบุ hardware/software ที่ใช้ทดลอง
assert sys.version_info >= (3, 9), (
    f"ต้องใช้ Python >= 3.9 (ตอนนี้ {platform.python_version()}) — ultralytics 8.4 ไม่รองรับต่ำกว่านี้")

def get_ram_gb():
    try:
        import psutil
        return round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        pass
    try:  # Linux fallback (ไม่มี psutil)
        return round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
    except (AttributeError, ValueError, OSError):
        return None

def list_kernelspecs():
    # คืน list ของ kernel ทุกตัวที่ JupyterLab เห็น: (internal name, display name, python ที่ใช้, ใช่ตัวที่รันอยู่ไหม)
    # ทำไม: ชื่อในเมนู (display) กับชื่อภายใน (name) ต่างกัน — ต้องรู้ชื่อภายในเพื่อผูก notebook ให้เปิดด้วย kernel ที่ถูกต้อง
    rows = []
    try:
        from jupyter_client.kernelspec import KernelSpecManager
        exe = Path(sys.executable).resolve()
        ksm = KernelSpecManager()
        for name in sorted(ksm.find_kernel_specs()):
            try:
                spec = ksm.get_kernel_spec(name)
                argv0 = Path(spec.argv[0])
                is_me = argv0.is_absolute() and argv0.resolve() == exe
                rows.append((name, spec.display_name, spec.argv[0], is_me))
            except Exception:
                continue
    except Exception:
        pass
    return rows

KERNELS = list_kernelspecs()
print("  kernels ที่ JupyterLab เห็น (internal name | display name | python):")
for name, disp, argv0, is_me in KERNELS:
    print(f"    {'>>' if is_me else '  '} {name:<22} | {disp:<36} | {argv0}")

def detect_kernel_display_names():
    return [disp for _, disp, _, is_me in KERNELS if is_me]

disk = shutil.disk_usage(PROJECT_ROOT)
sysinfo = {
    "python": platform.python_version(),
    "python_executable": sys.executable,
    "os": f"{platform.system()} {platform.release()}",
    "machine": platform.machine(),
    "hostname": platform.node(),
    "cpu_threads": os.cpu_count(),
    "ram_gb": get_ram_gb(),
    "disk_total_gb": round(disk.total / 1e9, 1),
    "disk_free_gb": round(disk.free / 1e9, 1),
    "kernel_display_names": detect_kernel_display_names(),
}
for k, v in sysinfo.items():
    print(f"  {k:<22}: {v}")

MIN_FREE_GB = 15   # dataset ดิบ 2 ชุด + interim/processed copy + runs/ ของ 2 experiments
if sysinfo["disk_free_gb"] < MIN_FREE_GB:
    print(f"[WARN] พื้นที่ว่าง {sysinfo['disk_free_gb']} GB < {MIN_FREE_GB} GB — ควรเคลียร์ก่อนดาวน์โหลด dataset ใน nb02")
else:
    print(f"[OK] พื้นที่ว่าง {sysinfo['disk_free_gb']} GB (>= {MIN_FREE_GB} GB)")

EXPECTED_KERNEL = "ML Environment (ds69) + YOLO26"   # venv ~/.venvs/yolo26 (ultralytics 8.4.155 + torch cu130) ที่ใช้จริงตั้งแต่ 2026-09-19
ACCEPTED_KERNELS = {EXPECTED_KERNEL, "ML Environment (ds69)"}  # conda ml_env ก็ใช้ได้ถ้าติดตั้ง ultralytics แล้ว
kn = sysinfo["kernel_display_names"]
if not kn:
    print(f"[INFO] ระบุชื่อ kernel อัตโนมัติไม่ได้ — เช็คมุมขวาบนของ JupyterLab ว่าเป็น '{EXPECTED_KERNEL}'")
elif not (ACCEPTED_KERNELS & set(kn)):
    print(f"[WARN] kernel ที่รันอยู่คือ {kn} ไม่ใช่ '{EXPECTED_KERNEL}'")
    print(f"       → เมนู Kernel ▸ Change Kernel… เลือก '{EXPECTED_KERNEL}' แล้วรันใหม่ตั้งแต่ CELL 1 (อย่ารันต่อ)")
else:
    print(f"[OK] kernel = {sorted(ACCEPTED_KERNELS & set(kn))} ({sys.executable})")
ENV_REPORT["system"] = sysinfo


In [ ]:
# === CELL 3 === ติดตั้ง/verify ultralytics + dependencies และยืนยันว่ารองรับ YOLO26 จริง
# ทำอะไร : เช็คว่า environment นี้เขียนได้ไหม → ติดตั้งเฉพาะที่ขาด → ถ้าเขียนไม่ได้ (env กลางของ JupyterHub เป็น read-only)
#          สร้าง venv ส่วนตัวใน home ที่ "มองเห็น package ของ ds69 ทั้งหมด" (torch/cv2/...) แล้ว register เป็น kernel ใหม่
#          → assert ว่า API ของ YOLO26 (end-to-end head, MuSGD, ProgLoss, nms=) มีครบ
# ทำไม   : /opt/jupyterhub_env เป็นของแอดมิน (Permission denied) และ --user ใช้ในvenv ไม่ได้ → ทางเดียวที่ทำเองได้คือ
#          venv ซ้อนที่ inherit package เดิม (ไม่ต้องโหลด torch ใหม่) ส่วนชื่อ argument ใน spec (end2end=, half=)
#          ถูก ultralytics รุ่นปัจจุบันเปลี่ยนเป็น nms= / quantize= → ต้อง assert ตรงนี้ ไม่งั้น nb05–07 benchmark ผิด head
import sysconfig
MIN_ULTRALYTICS = "8.4.154"                       # เวอร์ชันที่ทดสอบ API ทั้งหมดของ notebook ชุดนี้แล้ว
VENV_DIR = Path.home() / ".venvs" / "yolo26"      # venv ส่วนตัว (ใช้เฉพาะเมื่อ env ปัจจุบันเขียนไม่ได้)
KERNEL_NAME, KERNEL_DISPLAY = "yolo26-ds69", "ML Environment (ds69) + YOLO26"
ALLOW_FULL_INSTALL = False   # True เฉพาะเมื่อยืนยันแล้วว่า ds69 ไม่มี torch จริง ๆ และยอมให้ pip โหลด torch (หลาย GB) ลง venv ส่วนตัว

SITE_PKGS = Path(sysconfig.get_paths()["purelib"])
SITE_WRITABLE = os.access(SITE_PKGS, os.W_OK)
IN_VENV = sys.prefix != sys.base_prefix
print(f"  python        : {sys.executable}")
print(f"  site-packages : {SITE_PKGS}  (writable={SITE_WRITABLE}, venv={IN_VENV})")

def vtuple(v: str):
    # '8.4.154' → (8, 4, 154) เทียบเวอร์ชันโดยไม่พึ่ง package เสริม
    return tuple(int("".join(ch for ch in p if ch.isdigit()) or 0) for p in v.split("."))

def run(cmd, **kw):
    print("  $", " ".join(str(c) for c in cmd))
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print("    [pip/venv error]", (r.stderr or r.stdout).strip()[-1500:])
    return r.returncode == 0

def pip_install(*pkgs: str) -> bool:
    # ติดตั้งลง environment ปัจจุบัน (ลอง --user ด้วยถ้าไม่ได้อยู่ใน venv)
    if run([sys.executable, "-m", "pip", "install", "-q", *pkgs]):
        return True
    return (not IN_VENV) and run([sys.executable, "-m", "pip", "install", "-q", "--user", *pkgs])

def bootstrap_user_venv(pkgs) -> None:
    # 1) สร้าง ~/.venvs/yolo26 (idempotent)  2) ให้มองเห็น site-packages ของ env ปัจจุบันผ่านไฟล์ .pth
    # 3) pip install pkgs ลง venv ใหม่ (torch ฯลฯ ของ ds69 ถูกนับว่า "มีแล้ว" → ไม่โหลดซ้ำ)  4) register kernel
    is_win = platform.system() == "Windows"
    venv_py = VENV_DIR / ("Scripts/python.exe" if is_win else "bin/python")
    if not venv_py.exists():
        print(f"[VENV] สร้าง {VENV_DIR}")
        if not run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV_DIR)]):
            # ensurepip อาจไม่มี (Debian/Ubuntu) → สร้างแบบไม่มี pip แล้วใช้ pip ของ env เดิมชี้เข้าไปแทน
            assert run([sys.executable, "-m", "venv", "--system-site-packages", "--without-pip", str(VENV_DIR)]), \
                "สร้าง venv ไม่สำเร็จ — ส่ง error ด้านบนให้แอดมิน หรือขอให้ติดตั้ง ultralytics ลง ds69 โดยตรง"
    assert venv_py.exists(), f"ไม่พบ {venv_py}"
    # .pth: venv ซ้อน venv จะไม่เห็น site-packages ของ env กลาง (system-site-packages ชี้ไป python ฐานจริง) → เพิ่มเอง
    venv_site = subprocess.run([str(venv_py), "-c", "import sysconfig;print(sysconfig.get_paths()['purelib'])"],
                               capture_output=True, text=True).stdout.strip()
    Path(venv_site).mkdir(parents=True, exist_ok=True)
    (Path(venv_site) / "_ds69_base.pth").write_text(f"{SITE_PKGS}\n{sysconfig.get_paths()['platlib']}\n", encoding="utf-8")
    print(f"[VENV] เชื่อม package ของ ds69: {venv_site}/_ds69_base.pth → {SITE_PKGS}")
    # ติดตั้ง: ใช้ pip ใน venv ถ้ามี ไม่มีก็ใช้ pip เดิมแบบ --python (pip >= 22.3)
    ok = run([venv_py, "-m", "pip", "install", "-q", *pkgs]) or \
         run([sys.executable, "-m", "pip", "--python", venv_py, "install", "-q", *pkgs])
    assert ok, f"ติดตั้ง {list(pkgs)} ลง {VENV_DIR} ไม่สำเร็จ — ดู error ด้านบน"
    # register kernel (ipykernel มาจาก env ของ ds69 ผ่าน .pth)
    assert run([venv_py, "-m", "ipykernel", "install", "--user", "--name", KERNEL_NAME, "--display-name", KERNEL_DISPLAY]), \
        "register kernel ไม่สำเร็จ"
    print(f"[VENV] register kernel '{KERNEL_DISPLAY}' แล้ว")

def ensure_installed(*pkgs: str) -> None:
    # ติดตั้งลง env ปัจจุบันถ้าเขียนได้ ไม่งั้น bootstrap venv ส่วนตัวแล้วหยุดให้สลับ kernel (module ใหม่ต้องเริ่ม kernel ใหม่)
    if SITE_WRITABLE or not IN_VENV:
        if pip_install(*pkgs):
            importlib.invalidate_caches()
            return
    core_missing = [p for p in pkgs if p in ("torch", "torchvision", "numpy")]
    if core_missing and not ALLOW_FULL_INSTALL:
        # env นี้ไม่มีแม้แต่ torch/numpy → แทบแน่นอนว่าเปิดผิด kernel (เช่น 'Python 3 (ipykernel)' ที่เป็น env เปล่าของ JupyterHub)
        # ถ้าสร้าง venv จากตรงนี้จะไม่ได้ inherit อะไรเลย และ pip จะโหลด torch ใหม่หลาย GB → หยุดให้ตรวจ kernel ก่อน
        raise RuntimeError(
            f"environment นี้ ({sys.executable}) ไม่มี {core_missing} เลย → น่าจะเปิดผิด kernel (ดูมุมขวาบน)\n"
            f"  → Kernel ▸ Change Kernel… เลือก 'ML Environment (ds69)' แล้วรันใหม่ตั้งแต่ CELL 1\n"
            f"  → ถ้าใน ds69 ก็ไม่มี torch จริง ๆ ให้ตั้ง ALLOW_FULL_INSTALL = True ใน cell นี้แล้วรันอีกครั้ง")
    print(f"[FALLBACK] เขียน {SITE_PKGS} ไม่ได้ → สร้าง venv ส่วนตัวที่ inherit package ของ ds69 แล้วติดตั้ง {list(pkgs)} ที่นั่น")
    bootstrap_user_venv(pkgs)
    raise RuntimeError(
        f"ติดตั้งลง {VENV_DIR} แล้ว → Kernel ▸ Change Kernel… ▸ เลือก '{KERNEL_DISPLAY}' "
        f"(ถ้ายังไม่เห็น ให้ refresh หน้า JupyterLab) แล้วรันใหม่ตั้งแต่ CELL 1")

# 3.1 dependencies ที่ nb02–07 ต้องใช้ (import-name → pip-name) — เช็คก่อนเพื่อรวมติดตั้งรอบเดียว -----------
DEPS = {
    "torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python",
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "sklearn": "scikit-learn", "imagehash": "ImageHash", "yaml": "PyYAML",
    "tqdm": "tqdm", "psutil": "psutil", "PIL": "pillow", "lap": "lap",   # lap = ByteTrack assignment (nb07)
}
missing = [pkg for mod, pkg in DEPS.items() if importlib.util.find_spec(mod) is None]

# 3.2 ultralytics --------------------------------------------------------------------------------------
try:
    import ultralytics
    installed = ultralytics.__version__
except ImportError:
    installed = None

if installed is None:
    print(f"[INSTALL] ไม่พบ ultralytics → ติดตั้ง ultralytics>={MIN_ULTRALYTICS}" + (f" + {missing}" if missing else ""))
    ensure_installed(f"ultralytics>={MIN_ULTRALYTICS}", *missing)
    import ultralytics
    installed = ultralytics.__version__
    missing = [pkg for mod, pkg in DEPS.items() if importlib.util.find_spec(mod) is None]
elif vtuple(installed) < vtuple(MIN_ULTRALYTICS):
    print(f"[UPGRADE] ultralytics {installed} < {MIN_ULTRALYTICS}")
    ensure_installed("-U", f"ultralytics>={MIN_ULTRALYTICS}", *missing)
    raise RuntimeError("อัปเกรดแล้ว → Kernel ▸ Restart Kernel แล้วรันใหม่ตั้งแต่ CELL 1 (module เก่ายังค้างใน memory)")
print(f"[OK] ultralytics {installed} (tested with {MIN_ULTRALYTICS})")

if missing:
    print(f"[INSTALL] ขาด: {missing}")
    ensure_installed(*missing)
    still = [pkg for mod, pkg in DEPS.items() if importlib.util.find_spec(mod) is None]
    assert not still, f"ติดตั้งแล้วแต่ยัง import ไม่ได้: {still} → Restart Kernel แล้วรันใหม่"
print(f"[OK] dependencies ครบ {len(DEPS)} ตัว" + (f" (ติดตั้งเพิ่ม {missing})" if missing else ""))

# 3.3 YOLO26 feature asserts ---------------------------------------------------------------------------
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
cfg = get_cfg()
assert hasattr(cfg, "nms"), "ultralytics รุ่นนี้ไม่มี argument nms= → ต้องอัปเกรด"
_tmp = YOLO("yolo26s.yaml")                        # สร้างจาก yaml (ไม่ดาวน์โหลด) เพื่อดูโครงสร้าง head
_head = _tmp.model.model[-1]
assert _tmp.model.yaml.get("end2end") is True, "yolo26s.yaml ไม่ได้ตั้ง end2end: True → ไม่ใช่ YOLO26 จริง"
assert _tmp.model.yaml.get("reg_max") == 1, "YOLO26 ต้อง reg_max=1 (เลิกใช้ DFL แบบ v8)"
assert hasattr(_head, "one2one_cv2"), "head ไม่มี one2one branch → NMS-free inference ใช้ไม่ได้"
from ultralytics.optim import MuSGD            # optimizer ของ YOLO26 (Muon + SGD hybrid)
from ultralytics.utils.loss import E2ELoss     # ProgLoss: one2many → one2one progressive weighting
del _tmp, _head

api_facts = {
    "ultralytics_version": installed,
    "python_prefix": sys.prefix, "site_packages_writable": SITE_WRITABLE, "in_venv": IN_VENV,
    "nms_arg_default": cfg.nms,            # None = one-to-many head + NMS ภายนอก (ค่า default!)
    "has_quantize_arg": hasattr(cfg, "quantize"),
    "has_half_arg": hasattr(cfg, "half"),
    "default_tracker": cfg.tracker,
    "default_optimizer": cfg.optimizer,
    "spec_to_api": {
        "end2end=True (spec)": "nms=False  → NMS-free one-to-one head (deploy / benchmark ใน nb06–07)",
        "nms default":         "nms=None   → one-to-many + NMS ภายนอก (ใช้เทียบ NMS vs NMS-free ใน nb06)",
        "half=True (spec)":    "quantize=16 (FP16)" if hasattr(cfg, "quantize") else "half=True",
        "optimizer MuSGD":     "ต้องส่ง optimizer='MuSGD' ตรง ๆ ('auto' เลือกให้เฉพาะ iterations > 10k)",
        "ByteTrack (nb07)":    f"ต้องส่ง tracker='bytetrack.yaml' (default = {cfg.tracker})",
    },
}
print("[OK] YOLO26 support: end2end head + one2one branch + MuSGD + E2ELoss(ProgLoss) ✔")
print("     spec → installed API mapping:")
for k, v in api_facts["spec_to_api"].items():
    print(f"       {k:<22} : {v}")
ENV_REPORT["api"] = api_facts


In [ ]:
# === CELL 4 === ตรวจ GPU/CUDA → เลือก DEVICE (fallback CPU)
# ทำอะไร : เช็ค torch.cuda, ชื่อ GPU, VRAM, ทดสอบคูณ matrix บน GPU จริง 1 ครั้ง แล้วกำหนด DEVICE ให้ทุก notebook ใช้
# ทำไม   : เทรน 150 epochs บน CPU ใช้เวลาหลายวัน — ต้องรู้ตั้งแต่ตอนนี้ ไม่ใช่ตอน nb05 และ driver/CUDA ที่ไม่เข้ากัน
#          จะโผล่ตอน tensor แรกแตะ GPU → จับตรงนี้ดีกว่าจับกลางการเทรน
import torch
set_seed(SEED)   # เรียกซ้ำหลัง torch พร้อมใช้ ให้แน่ใจว่า torch.manual_seed ถูกตั้ง

gpu_info = {"torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda, "cudnn": torch.backends.cudnn.version()}
DEVICE = "cpu"
if gpu_info["cuda_available"]:
    try:
        props = torch.cuda.get_device_properties(0)
        gpu_info.update({"gpu_name": props.name, "vram_gb": round(props.total_memory / 1e9, 2),
                         "compute_capability": f"{props.major}.{props.minor}",
                         "gpu_count": torch.cuda.device_count()})
        a = torch.randn(2048, 2048, device="cuda")
        b = torch.randn(2048, 2048, device="cuda")
        torch.cuda.synchronize(); t0 = time.perf_counter()
        (a @ b).sum().item(); torch.cuda.synchronize()
        gpu_info["matmul_2048_ms"] = round((time.perf_counter() - t0) * 1000, 2)
        del a, b; torch.cuda.empty_cache()
        DEVICE = 0
        print(f"[OK] GPU: {props.name} | VRAM {gpu_info['vram_gb']} GB | CUDA {gpu_info['cuda_version']} "
              f"| cuDNN {gpu_info['cudnn']} | matmul 2048x2048 = {gpu_info['matmul_2048_ms']} ms")
        if gpu_info["vram_gb"] < 6:
            print(f"[WARN] VRAM {gpu_info['vram_gb']} GB ค่อนข้างน้อย → nb05 ใช้ batch=-1 (AutoBatch) + cache=False, "
                  f"และคาดว่า batch จะได้ราว 8–16 ที่ imgsz=640")
    except Exception as e:
        print(f"[ERROR] CUDA มองเห็นแต่ใช้งานไม่ได้: {e}\n        → fallback เป็น CPU")
        DEVICE = "cpu"
else:
    print(f"[WARN] ไม่พบ CUDA (torch {torch.__version__}) → ใช้ CPU: smoke test ได้ แต่เทรน 150 epochs จะช้ามาก")
    print("       วิธีแก้: ติดตั้ง torch รุ่น CUDA ให้ตรงกับ driver ของเครื่อง → https://pytorch.org/get-started/locally/")
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=10)
        if smi.returncode == 0:
            print(f"       nvidia-smi เห็น GPU: {smi.stdout.strip()} (มี driver แต่ torch เป็นรุ่น CPU)")
    except Exception:
        pass
gpu_info["device_selected"] = DEVICE
ENV_REPORT["gpu"] = gpu_info
print(f"[OK] DEVICE = {DEVICE!r}")


In [ ]:
# === CELL 5 === สร้างโครงโฟลเดอร์โปรเจกต์ (idempotent) + เขียน configs/project.yaml
# ทำอะไร : mkdir ทุกโฟลเดอร์ใน CONFIG['dirs'] (รันซ้ำได้ ไม่ทับของเดิม) แล้ว dump CONFIG เป็น YAML + อ่านกลับตรวจ
# ทำไม   : nb02–07 จะ load configs/project.yaml เป็น single source of truth (path / classes / hyper-params / device)
import yaml

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
(DIRS["src"] / "__init__.py").touch(exist_ok=True)

CONFIG["device"] = DEVICE
CONFIG["updated_at"] = datetime.now().isoformat(timespec="seconds")
cfg_path = DIRS["configs"] / "project.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

def print_tree(root: Path, max_depth=2, _prefix="", _depth=0):
    # แสดงเฉพาะโฟลเดอร์ (ลึกไม่เกิน max_depth) ให้เห็นโครงสร้างในเอาต์พุต
    if _depth > max_depth:
        return
    kids = sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith("."))
    for i, p in enumerate(kids):
        last = i == len(kids) - 1
        print(f"{_prefix}{'└── ' if last else '├── '}{p.name}/")
        print_tree(p, max_depth, _prefix + ("    " if last else "│   "), _depth + 1)

print(f"[OK] สร้าง {len(DIRS)} โฟลเดอร์ + เขียน {cfg_path.relative_to(PROJECT_ROOT)}")
print(f"{PROJECT_ROOT.name}/")
print_tree(PROJECT_ROOT)

with open(cfg_path, encoding="utf-8") as f:
    _chk = yaml.safe_load(f)
assert _chk["classes"] == CONFIG["classes"] and _chk["seed"] == SEED, "project.yaml เขียน/อ่านกลับไม่ตรงกัน"
assert list(_chk["classes"].keys()) == [0, 1, 2, 3], "class index ใน YAML ต้องเป็น 0–3 ตามลำดับ"
print("[OK] อ่าน project.yaml กลับได้ครบ: classes/seed/dirs ตรงกับ CONFIG")


In [ ]:
# === CELL 6 === ดาวน์โหลด yolo26s.pt + smoke test 2 โหมด (NMS default vs NMS-free)
# ทำอะไร : โหลด pretrained COCO ลง weights/ → predict ภาพตัวอย่างของ ultralytics ด้วย nms=None และ nms=False
# ทำไม   : ยืนยันว่า (1) โมเดลโหลด/รันได้บน DEVICE ที่เลือก (2) ทั้ง 2 head ทำงานจริง — nb06 ต้องเทียบ NMS vs NMS-free
#          และเก็บ ms/frame เบื้องต้นไว้เป็น sanity baseline (benchmark จริงทำใน nb06 บน test set)
from ultralytics import YOLO
from ultralytics.utils import ASSETS
from ultralytics.utils.downloads import attempt_download_asset
import cv2

weights_path = DIRS["weights"] / CONFIG["model"]["base_weights"]
if not weights_path.exists():
    print(f"[DOWNLOAD] {weights_path.name} → {weights_path.parent.relative_to(PROJECT_ROOT)}/")
    attempt_download_asset(str(weights_path))     # ดาวน์โหลดจาก GitHub release ของ ultralytics ลง path นี้โดยตรง
assert weights_path.exists() and weights_path.stat().st_size > 10_000_000, (
    f"ไฟล์ {weights_path} หาย/ไม่สมบูรณ์ — ลบไฟล์แล้วรัน cell นี้ใหม่")

model = YOLO(str(weights_path))
n_params = sum(p.numel() for p in model.model.parameters())
model_info = {"weights": str(weights_path.relative_to(PROJECT_ROOT)),
              "size_mb": round(weights_path.stat().st_size / 1e6, 2),
              "params_M": round(n_params / 1e6, 2),
              "nc_pretrained": model.model.nc, "task": model.task}
assert model_info["nc_pretrained"] == 80, "pretrained COCO ต้องมี 80 คลาส"
print(f"[OK] โหลด {weights_path.name}: {model_info['params_M']}M params, {model_info['size_mb']} MB, "
      f"nc={model_info['nc_pretrained']} (COCO), task={model_info['task']}")

sample_img = ASSETS / "bus.jpg"     # ภาพตัวอย่างที่มากับ ultralytics (ไม่ต้องดาวน์โหลดเพิ่ม)
N_WARMUP, N_RUNS = 3, 10
smoke = {}
for mode, nms_flag in [("nms_default", None), ("nms_free", False)]:
    for _ in range(N_WARMUP):
        model.predict(sample_img, imgsz=640, device=DEVICE, nms=nms_flag, verbose=False)
    speeds = []
    for _ in range(N_RUNS):
        r = model.predict(sample_img, imgsz=640, device=DEVICE, nms=nms_flag, verbose=False)[0]
        speeds.append(r.speed)
    assert model.predictor.args.nms is nms_flag, f"predictor ไม่ได้สลับเป็น nms={nms_flag}"
    assert len(r.boxes) > 0, f"โหมด {mode} ตรวจไม่เจออะไรใน bus.jpg เลย — โมเดล/device ผิดปกติ"
    m = {
        "nms_arg": nms_flag,
        "detections": len(r.boxes),
        "preprocess_ms":  round(sum(s["preprocess"]  for s in speeds) / N_RUNS, 2),
        "inference_ms":   round(sum(s["inference"]   for s in speeds) / N_RUNS, 2),
        "postprocess_ms": round(sum(s["postprocess"] for s in speeds) / N_RUNS, 2),
    }
    m["total_ms"] = round(m["preprocess_ms"] + m["inference_ms"] + m["postprocess_ms"], 2)
    m["fps"] = round(1000 / m["total_ms"], 1)
    out_img = DIRS["figures"] / f"01_smoke_{mode}.jpg"
    cv2.imwrite(str(out_img), r.plot())            # ภาพผลตรวจจับ (สำหรับดูด้วยตา ไม่ใช่กราฟ)
    smoke[mode] = m
    print(f"[OK] {mode:<12} nms={str(nms_flag):<5} dets={m['detections']:<3} "
          f"pre {m['preprocess_ms']:>6.2f} | infer {m['inference_ms']:>7.2f} | post {m['postprocess_ms']:>6.2f} ms "
          f"→ {m['total_ms']:>7.2f} ms/frame ({m['fps']} FPS)  → {out_img.name}")

print(f"     device={DEVICE!r} | imgsz=640 | mean of {N_RUNS} runs after {N_WARMUP} warm-up | "
      f"NMS-free ควรมี postprocess ต่ำกว่า default ชัดเจน (ไม่มีขั้น NMS)")
ENV_REPORT["model"] = model_info
ENV_REPORT["smoke_test"] = smoke


In [ ]:
# === CELL 7 === บันทึก reports/env_report.json + requirements.txt + สรุป
# ทำอะไร : รวมทุกอย่างที่ตรวจได้ (ระบบ / API / GPU / model / smoke test) ลง JSON และ pin เวอร์ชัน package ที่ติดตั้งจริง
# ทำไม   : ใช้อ้างอิงในเล่มรายงาน (hardware/software ที่ใช้ทดลอง) และให้ทีมสร้าง environment เดียวกันซ้ำได้
from importlib.metadata import version as pkg_version, PackageNotFoundError

PIN_PACKAGES = ["ultralytics", "torch", "torchvision", "opencv-python", "numpy", "pandas", "matplotlib",
                "seaborn", "scikit-learn", "ImageHash", "PyYAML", "tqdm", "psutil", "pillow", "lap"]
pinned, not_found = [], []
for pkg in PIN_PACKAGES:
    try:
        pinned.append(f"{pkg}=={pkg_version(pkg)}")
    except PackageNotFoundError:
        not_found.append(pkg)
req_path = PROJECT_ROOT / "requirements.txt"
req_path.write_text(f"# generated by 01_setup_env.ipynb on {datetime.now():%Y-%m-%d %H:%M} "
                    f"({sysinfo['os']}, Python {sysinfo['python']})\n" + "\n".join(pinned) + "\n", encoding="utf-8")
ENV_REPORT["requirements"] = pinned
ENV_REPORT["finished_at"] = datetime.now().isoformat(timespec="seconds")

report_path = DIRS["reports"] / "env_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(ENV_REPORT, f, ensure_ascii=False, indent=2, default=str)

print(f"[OK] {req_path.name}: pin {len(pinned)} packages" + (f" (ไม่พบ: {not_found})" if not_found else ""))
print(f"[OK] {report_path.relative_to(PROJECT_ROOT)}")
print("\n=== SUMMARY · Notebook 01 ===")
print(f"  Python {sysinfo['python']} | {sysinfo['os']} | CPU {sysinfo['cpu_threads']} threads "
      f"| RAM {sysinfo['ram_gb']} GB | disk free {sysinfo['disk_free_gb']} GB")
print(f"  ultralytics {api_facts['ultralytics_version']} | torch {gpu_info['torch']} | DEVICE = {DEVICE!r}"
      + (f" ({gpu_info['gpu_name']}, {gpu_info['vram_gb']} GB)" if DEVICE != "cpu" else " (CPU — ดู [WARN] ใน CELL 4)"))
print(f"  {CONFIG['model']['base_weights']}: {model_info['params_M']}M params | smoke NMS-free "
      f"{smoke['nms_free']['total_ms']} ms/frame vs default {smoke['nms_default']['total_ms']} ms/frame")
print(f"  config → {cfg_path.relative_to(PROJECT_ROOT)} | ถัดไป: 02_data_merge_mapping.ipynb")


## ผลลัพธ์ที่ควรเห็น
- ทุก cell ขึ้น `[OK]` ไม่มี `AssertionError`; CELL 4 ต้องเห็นชื่อ GPU (ถ้าขึ้น `[WARN] ไม่พบ CUDA` ให้แก้ก่อนไป nb05)
- CELL 6: `nms_free` มี **postprocess ต่ำกว่า `nms_default` ชัดเจน** (ไม่มีขั้น NMS) — จำนวน detections อาจต่างกันเล็กน้อยได้
- ไฟล์ที่เกิดขึ้น: `configs/project.yaml`, `weights/yolo26s.pt`, `figures/01_smoke_*.jpg`, `reports/env_report.json`, `requirements.txt`

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| `[UPGRADE] ... Restart Kernel` | ultralytics ใน ds69 เก่ากว่า 8.4.154 | Kernel ▸ Restart Kernel แล้วรันใหม่ตั้งแต่ CELL 1 |
| `[FALLBACK] ... RuntimeError: ติดตั้งลง ~/.venvs/yolo26 แล้ว` | `/opt/jupyterhub_env` เป็น read-only (Permission denied) และ `--user` ใช้ใน venv ไม่ได้ | **ไม่ใช่ error จริง** — cell สร้าง venv ส่วนตัวที่ inherit package ของ ds69 + register kernel ใหม่ให้แล้ว → Kernel ▸ Change Kernel… ▸ `ML Environment (ds69) + YOLO26` (refresh หน้าเว็บถ้ายังไม่เห็น) แล้วรันใหม่ตั้งแต่ CELL 1 |
| อยากใช้ kernel `ML Environment (ds69)` เดิมเป๊ะ ๆ | ultralytics ไม่ได้ติดตั้งใน env กลาง | ขอแอดมินรัน `/opt/jupyterhub_env/bin/python3 -m pip install "ultralytics>=8.4.154" ImageHash lap` → notebook จะตรวจเจอเองและไม่สร้าง venv |
| `[WARN] ไม่พบ CUDA` | torch เป็นรุ่น CPU หรือ driver เก่า | ติดตั้ง torch รุ่น CUDA ให้ตรง driver (ดูคำสั่งที่ pytorch.org) แล้วรัน CELL 4 ใหม่ |
| ดาวน์โหลด `yolo26s.pt` ค้าง | เซิร์ฟเวอร์ปิด outbound ไป github.com | ดาวน์โหลดจากเครื่องอื่นแล้ว upload ไปที่ `weights/yolo26s.pt` แล้วรัน CELL 6 ใหม่ |
| `RuntimeError: environment นี้ ... ไม่มี ['torch', ...] เลย → น่าจะเปิดผิด kernel` | notebook ถูกเปิดด้วย `Python 3 (ipykernel)` (env เปล่าของ JupyterHub) ไม่ใช่ `ML Environment (ds69)` | Kernel ▸ Change Kernel… ▸ `ML Environment (ds69)` แล้วรันใหม่ตั้งแต่ CELL 1 — ดูตาราง kernels ใน CELL 2 เพื่อยืนยัน |
| `[WARN] cwd ไม่ใช่ YOLOV26` | เปิด notebook จากโฟลเดอร์อื่น | ย้าย notebook ไปไว้ใน `GROUP NI YOD YIAM MAK MAK/YOLOV26/` แล้วเปิดใหม่ |
